<a href="https://colab.research.google.com/github/Fajtos/hzmps-surveillance/blob/main/real_dengue_hzmps_baseline_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline Model Comparison for HZMPS Dengue Application

**Purpose:** Compare the proposed HZMPS state-space model against three
standard alternatives on the same dengue surveillance data.

**Data:** 50 municipalities in São Paulo state, last 104 weeks
(2022-W01 to 2023-W52).

**Models compared:**
1. HZMPS-SS (already fitted): Hurdle Zero-Modified Power Series + sparse
   finite mixture over units + AR(1) count mean.
2. ZIP-SS: Zero-inflated Poisson + same mixture + same AR(1).
3. ZINB-SS: Zero-inflated Negative Binomial + same mixture + same AR(1).
4. HNB-SS: Hurdle Negative Binomial + same mixture + same AR(1).

**Comparison metric:** WAIC (Watanabe-Akaike Information Criterion).
Lower is better.

**Fairness:** All four models use identical state-space structure, identical
mixture prior, identical AR(1) prior. Only the observation distribution differs.

**Expected runtime:** ~6 hours total across 3 new fits.

In [ ]:
# CELL 2: SETUP AND DATA RELOAD
!pip install --upgrade cmdstanpy -q

import os, urllib.request, shutil
CMDSTAN_DIR = '/content/cmdstan-2.36.0'
if not os.path.exists(os.path.join(CMDSTAN_DIR, 'makefile')):
    os.chdir('/content')
    tgz_file = '/content/colab-cmdstan-2.36.0.tar.gz'
    tgz_url = ('https://github.com/stan-dev/cmdstan/releases/download/'
               'v2.36.0/colab-cmdstan-2.36.0.tgz')
    if not os.path.exists(tgz_file):
        urllib.request.urlretrieve(tgz_url, tgz_file)
    shutil.unpack_archive(tgz_file, '/content')
os.environ['CMDSTAN'] = CMDSTAN_DIR

import cmdstanpy, numpy as np, pandas as pd, requests, io, time
print(f"CmdStan: {cmdstanpy.cmdstan_version()}")

# --- Reload dengue panel (same 50 municipalities, last 104 weeks) ---
url = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/SP/municipios"
df_sp = pd.DataFrame([
    {'geocode': m['id'], 'name': m['nome']} for m in requests.get(url).json()
])

N_MUN = 50
np.random.seed(42)
selected = df_sp.sample(N_MUN, random_state=42).reset_index(drop=True)

all_data = []
for _, row in selected.iterrows():
    url_api = (f"https://info.dengue.mat.br/api/alertcity"
               f"?geocode={int(row['geocode'])}&disease=dengue&format=csv"
               f"&ew_start=1&ew_end=53&ey_start=2019&ey_end=2023")
    r = requests.get(url_api, timeout=30)
    df = pd.read_csv(io.StringIO(r.text))
    df['unit_id'] = int(row['geocode'])
    all_data.append(df)
    time.sleep(0.3)

df_raw = pd.concat(all_data, ignore_index=True)
df_raw['date'] = pd.to_datetime(df_raw['data_iniSE'])
df_raw['year'] = df_raw['date'].dt.year
df_raw['week'] = df_raw['date'].dt.isocalendar().week.astype(int)
df_raw['time_idx'] = df_raw['year'] * 100 + df_raw['week']

panel_full = df_raw.pivot_table(
    index='unit_id', columns='time_idx', values='casos', aggfunc='sum'
).fillna(0)

# Truncate to last 104 weeks
panel = panel_full.iloc[:, -104:]
Y = panel.values.astype(int)
N, T = panel.shape
M = N * T

print(f"Panel shape: N={N}, T={T}, M={M}")
print(f"Zero fraction: {(Y==0).mean():.3f}")

# Package as flat vectors for Stan
y_flat   = Y.flatten().tolist()
unit_idx = np.repeat(np.arange(1, N+1), T).tolist()
time_idx = np.tile(np.arange(1, T+1), N).tolist()

base_data = {
    'N': N, 'T': T, 'G': 5,
    'y': y_flat, 'unit': unit_idx, 'time': time_idx,
    'a_dirichlet': 0.5, 'delta_min': 0.3,
    'grainsize': max(1, int(M / 8)),
}
print("Base data prepared.")

CmdStan: (2, 36)
Panel shape: N=50, T=104, M=5200
Zero fraction: 0.456
Base data prepared.


In [ ]:
# CELL 3: ZIP STATE-SPACE MODEL (ZERO-INFLATED POISSON)
zip_code = r"""
functions {
  real partial_sum_zip(array[] int y_slice, int start, int end,
                       array[] int unit, array[] int time,
                       vector pi_i, matrix mu) {
    real lp = 0;
    for (k in 1:size(y_slice)) {
      int m = start + k - 1;
      int i = unit[m]; int t = time[m];
      if (y_slice[k] == 0) {
        lp += log_sum_exp(log(pi_i[i]), log1m(pi_i[i]) - mu[i, t]);
      } else {
        lp += log1m(pi_i[i])
              + y_slice[k] * log(mu[i, t]) - mu[i, t]
              - lgamma(y_slice[k] + 1);
      }
    }
    return lp;
  }
}
data {
  int<lower=1> N; int<lower=1> T; int<lower=1> G;
  array[N*T] int<lower=0> y;
  array[N*T] int<lower=1> unit;
  array[N*T] int<lower=1> time;
  real<lower=0> a_dirichlet; real<lower=0> delta_min;
  int<lower=1> grainsize;
}
transformed data { int<lower=0> M = N * T; }
parameters {
  real mu_eta1; vector[G-1] gamma_raw;
  real<lower=0> sigma_eta;
  vector[N] eta; simplex[G] w;
  real beta0_mu;
  real<lower=-0.99, upper=0.99> rho_zeta;
  real<lower=0> sigma_zeta;
  matrix[N, T] zeta_raw;
}
transformed parameters {
  ordered[G] mu_eta;
  mu_eta[1] = mu_eta1;
  for (g in 2:G) mu_eta[g] = mu_eta[g-1] + delta_min + log1p_exp(gamma_raw[g-1]);
  matrix[N, T] zeta;
  for (i in 1:N) {
    zeta[i, 1] = zeta_raw[i, 1] * sigma_zeta / sqrt(1 - square(rho_zeta));
    for (t in 2:T)
      zeta[i, t] = rho_zeta * zeta[i, t-1] + zeta_raw[i, t] * sigma_zeta;
  }
  vector[N] pi_i;   // zero-inflation probability
  for (i in 1:N) pi_i[i] = inv_logit(eta[i]);
  matrix[N, T] mu;
  for (i in 1:N) for (t in 1:T) mu[i, t] = exp(beta0_mu + zeta[i, t]);
}
model {
  mu_eta1    ~ normal(0, 3);
  gamma_raw  ~ normal(-1, 1);
  sigma_eta  ~ normal(0, 1);
  w          ~ dirichlet(rep_vector(a_dirichlet / G, G));
  beta0_mu   ~ normal(0, 5);
  rho_zeta   ~ uniform(-0.99, 0.99);
  sigma_zeta ~ normal(0, 1);
  to_vector(zeta_raw) ~ std_normal();
  for (i in 1:N) {
    vector[G] lp;
    for (g in 1:G) lp[g] = log(w[g]) + normal_lpdf(eta[i] | mu_eta[g], sigma_eta);
    target += log_sum_exp(lp);
  }
  target += reduce_sum(partial_sum_zip, y, grainsize, unit, time, pi_i, mu);
}
generated quantities {
  vector[M] log_lik;
  for (m in 1:M) {
    int i = unit[m]; int t = time[m];
    if (y[m] == 0)
      log_lik[m] = log_sum_exp(log(pi_i[i]), log1m(pi_i[i]) - mu[i, t]);
    else
      log_lik[m] = log1m(pi_i[i]) + y[m]*log(mu[i,t]) - mu[i,t]
                   - lgamma(y[m]+1);
  }
}
"""
with open('/content/zip_ss.stan', 'w') as f:
    f.write(zip_code)
print("ZIP Stan model written.")

t0 = time.time()
zip_model = cmdstanpy.CmdStanModel(
    stan_file='/content/zip_ss.stan',
    cpp_options={'STAN_THREADS': 'true'}, force_compile=True)
print(f"ZIP compiled in {time.time()-t0:.1f} s")

ZIP Stan model written.
ZIP compiled in 230.6 s


In [ ]:
# CELL 4: ZINB STATE-SPACE MODEL
zinb_code = r"""
functions {
  real partial_sum_zinb(array[] int y_slice, int start, int end,
                        array[] int unit, array[] int time,
                        vector pi_i, matrix mu, real phi) {
    real lp = 0;
    for (k in 1:size(y_slice)) {
      int m = start + k - 1;
      int i = unit[m]; int t = time[m];
      real mu_it = mu[i, t];
      real nb0 = phi * log(phi / (phi + mu_it));   // log NB(y=0)
      if (y_slice[k] == 0) {
        lp += log_sum_exp(log(pi_i[i]), log1m(pi_i[i]) + nb0);
      } else {
        real log_nb = lgamma(y_slice[k] + phi) - lgamma(phi)
                      - lgamma(y_slice[k] + 1)
                      + phi * log(phi / (phi + mu_it))
                      + y_slice[k] * log(mu_it / (phi + mu_it));
        lp += log1m(pi_i[i]) + log_nb;
      }
    }
    return lp;
  }
}
data {
  int<lower=1> N; int<lower=1> T; int<lower=1> G;
  array[N*T] int<lower=0> y;
  array[N*T] int<lower=1> unit;
  array[N*T] int<lower=1> time;
  real<lower=0> a_dirichlet; real<lower=0> delta_min;
  int<lower=1> grainsize;
}
transformed data { int<lower=0> M = N * T; }
parameters {
  real mu_eta1; vector[G-1] gamma_raw;
  real<lower=0> sigma_eta;
  vector[N] eta; simplex[G] w;
  real beta0_mu;
  real<lower=-0.99, upper=0.99> rho_zeta;
  real<lower=0> sigma_zeta;
  matrix[N, T] zeta_raw;
  real<lower=0> phi;
}
transformed parameters {
  ordered[G] mu_eta;
  mu_eta[1] = mu_eta1;
  for (g in 2:G) mu_eta[g] = mu_eta[g-1] + delta_min + log1p_exp(gamma_raw[g-1]);
  matrix[N, T] zeta;
  for (i in 1:N) {
    zeta[i, 1] = zeta_raw[i, 1] * sigma_zeta / sqrt(1 - square(rho_zeta));
    for (t in 2:T)
      zeta[i, t] = rho_zeta * zeta[i, t-1] + zeta_raw[i, t] * sigma_zeta;
  }
  vector[N] pi_i;
  for (i in 1:N) pi_i[i] = inv_logit(eta[i]);
  matrix[N, T] mu;
  for (i in 1:N) for (t in 1:T) mu[i, t] = exp(beta0_mu + zeta[i, t]);
}
model {
  mu_eta1    ~ normal(0, 3);
  gamma_raw  ~ normal(-1, 1);
  sigma_eta  ~ normal(0, 1);
  w          ~ dirichlet(rep_vector(a_dirichlet / G, G));
  beta0_mu   ~ normal(0, 5);
  rho_zeta   ~ uniform(-0.99, 0.99);
  sigma_zeta ~ normal(0, 1);
  phi        ~ normal(0, 5);   // half-normal prior on NB dispersion
  to_vector(zeta_raw) ~ std_normal();
  for (i in 1:N) {
    vector[G] lp;
    for (g in 1:G) lp[g] = log(w[g]) + normal_lpdf(eta[i] | mu_eta[g], sigma_eta);
    target += log_sum_exp(lp);
  }
  target += reduce_sum(partial_sum_zinb, y, grainsize, unit, time, pi_i, mu, phi);
}
generated quantities {
  vector[M] log_lik;
  for (m in 1:M) {
    int i = unit[m]; int t = time[m];
    real mu_it = mu[i, t];
    real log_nb = lgamma(y[m] + phi) - lgamma(phi) - lgamma(y[m] + 1)
                  + phi * log(phi / (phi + mu_it))
                  + y[m] * log(mu_it / (phi + mu_it));
    if (y[m] == 0)
      log_lik[m] = log_sum_exp(log(pi_i[i]),
                               log1m(pi_i[i]) + phi * log(phi / (phi + mu_it)));
    else
      log_lik[m] = log1m(pi_i[i]) + log_nb;
  }
}
"""
with open('/content/zinb_ss.stan', 'w') as f:
    f.write(zinb_code)
print("ZINB Stan model written.")

t0 = time.time()
zinb_model = cmdstanpy.CmdStanModel(
    stan_file='/content/zinb_ss.stan',
    cpp_options={'STAN_THREADS': 'true'}, force_compile=True)
print(f"ZINB compiled in {time.time()-t0:.1f} s")

ZINB Stan model written.
ZINB compiled in 30.5 s


In [ ]:
## CELL 5: HNB STATE-SPACE MODEL (HURDLE NEGATIVE BINOMIAL)
hnb_code = r"""
functions {
  real partial_sum_hnb(array[] int y_slice, int start, int end,
                       array[] int unit, array[] int time,
                       vector omega, matrix mu, real phi) {
    real lp = 0;
    for (k in 1:size(y_slice)) {
      int m = start + k - 1;
      int i = unit[m]; int t = time[m];
      real mu_it = mu[i, t];
      real nb0 = phi * log(phi / (phi + mu_it));
      real log_norm = log1m_exp(nb0);   // log(1 - NB(0))
      if (y_slice[k] == 0) {
        lp += log1m(omega[i]);
      } else {
        real log_nb = lgamma(y_slice[k] + phi) - lgamma(phi)
                      - lgamma(y_slice[k] + 1)
                      + phi * log(phi / (phi + mu_it))
                      + y_slice[k] * log(mu_it / (phi + mu_it));
        lp += log(omega[i]) + log_nb - log_norm;
      }
    }
    return lp;
  }
}
data {
  int<lower=1> N; int<lower=1> T; int<lower=1> G;
  array[N*T] int<lower=0> y;
  array[N*T] int<lower=1> unit;
  array[N*T] int<lower=1> time;
  real<lower=0> a_dirichlet; real<lower=0> delta_min;
  int<lower=1> grainsize;
}
transformed data { int<lower=0> M = N * T; }
parameters {
  real mu_eta1; vector[G-1] gamma_raw;
  real<lower=0> sigma_eta;
  vector[N] eta; simplex[G] w;
  real beta0_mu;
  real<lower=-0.99, upper=0.99> rho_zeta;
  real<lower=0> sigma_zeta;
  matrix[N, T] zeta_raw;
  real<lower=0> phi;
}
transformed parameters {
  ordered[G] mu_eta;
  mu_eta[1] = mu_eta1;
  for (g in 2:G) mu_eta[g] = mu_eta[g-1] + delta_min + log1p_exp(gamma_raw[g-1]);
  matrix[N, T] zeta;
  for (i in 1:N) {
    zeta[i, 1] = zeta_raw[i, 1] * sigma_zeta / sqrt(1 - square(rho_zeta));
    for (t in 2:T)
      zeta[i, t] = rho_zeta * zeta[i, t-1] + zeta_raw[i, t] * sigma_zeta;
  }
  vector[N] omega;
  for (i in 1:N) omega[i] = inv_logit(eta[i]);
  matrix[N, T] mu;
  for (i in 1:N) for (t in 1:T) mu[i, t] = exp(beta0_mu + zeta[i, t]);
}
model {
  mu_eta1    ~ normal(0, 3);
  gamma_raw  ~ normal(-1, 1);
  sigma_eta  ~ normal(0, 1);
  w          ~ dirichlet(rep_vector(a_dirichlet / G, G));
  beta0_mu   ~ normal(0, 5);
  rho_zeta   ~ uniform(-0.99, 0.99);
  sigma_zeta ~ normal(0, 1);
  phi        ~ normal(0, 5);
  to_vector(zeta_raw) ~ std_normal();
  for (i in 1:N) {
    vector[G] lp;
    for (g in 1:G) lp[g] = log(w[g]) + normal_lpdf(eta[i] | mu_eta[g], sigma_eta);
    target += log_sum_exp(lp);
  }
  target += reduce_sum(partial_sum_hnb, y, grainsize, unit, time, omega, mu, phi);
}
generated quantities {
  vector[M] log_lik;
  for (m in 1:M) {
    int i = unit[m]; int t = time[m];
    real mu_it = mu[i, t];
    real nb0 = phi * log(phi / (phi + mu_it));
    real log_norm = log1m_exp(nb0);
    real log_nb = lgamma(y[m] + phi) - lgamma(phi) - lgamma(y[m] + 1)
                  + phi * log(phi / (phi + mu_it))
                  + y[m] * log(mu_it / (phi + mu_it));
    if (y[m] == 0)
      log_lik[m] = log1m(omega[i]);
    else
      log_lik[m] = log(omega[i]) + log_nb - log_norm;
  }
}
"""
with open('/content/hnb_ss.stan', 'w') as f:
    f.write(hnb_code)
print("HNB Stan model written.")

t0 = time.time()
hnb_model = cmdstanpy.CmdStanModel(
    stan_file='/content/hnb_ss.stan',
    cpp_options={'STAN_THREADS': 'true'}, force_compile=True)
print(f"HNB compiled in {time.time()-t0:.1f} s")

HNB Stan model written.
HNB compiled in 29.4 s


In [ ]:
# CELL 6: FIT BASELINE MODELS (sequential)
# Each fit: 2 chains x 1000 warmup + 1000 sampling
# Expected: ~2 hours per fit, ~6 hours total.

def fit_baseline(model, name, stan_data, output_dir):
    print(f"\n{'='*60}")
    print(f"Fitting {name}...")
    print(f"{'='*60}")
    t0 = time.time()
    fit = model.sample(
        data=stan_data, chains=2, parallel_chains=2,
        threads_per_chain=2,
        iter_warmup=1000, iter_sampling=1000,
        seed=42, adapt_delta=0.98, max_treedepth=12,
        show_progress=True, output_dir=output_dir,
    )
    elapsed = (time.time() - t0) / 60
    print(f"{name} completed in {elapsed:.1f} min")
    fit.draws_pd().to_csv(f'/content/{name}_draws.csv', index=False)
    fit.summary().to_csv(f'/content/{name}_summary.csv')
    return fit

fit_zip  = fit_baseline(zip_model,  'zip_ss',  base_data, '/content/fit_zip')
fit_zinb = fit_baseline(zinb_model, 'zinb_ss', base_data, '/content/fit_zinb')
fit_hnb  = fit_baseline(hnb_model,  'hnb_ss',  base_data, '/content/fit_hnb')

print("\nAll baseline fits complete.")


Fitting zip_ss...


chain 1:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

	Chain 1 had 43 divergent transitions (4.3%)
	Chain 2 had 41 divergent transitions (4.1%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


zip_ss completed in 205.5 min

Fitting zinb_ss...


...

chain 1:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]

Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'hnb_ss.stan', line 72, column 19 to column 82)
Consider re-running with show_console=True if the above output is unclear!


	Chain 1 had 19 divergent transitions (1.9%)
	Chain 2 had 22 divergent transitions (2.2%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


hnb_ss completed in 276.6 min

All baseline fits complete.


In [ ]:
# Check if the refit can be done in this session
try:
    print("panel exists:", panel.shape)
except NameError:
    print("panel NOT loaded")
try:
    print("model_v2 exists:", model_v2)
except NameError:
    print("model_v2 NOT compiled")

panel exists: (50, 104)
model_v2 NOT compiled


In [ ]:
import os

files_to_check = [
    '/content/zip_ss_draws.csv',
    '/content/zinb_ss_draws.csv',
    '/content/zip_ss_summary.csv',
    '/content/zinb_ss_summary.csv',
    '/content/dengue_draws_v3.csv',
]

for f in files_to_check:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / 1e6
        print(f"✅ {f}  ({size_mb:.1f} MB)")
    else:
        print(f"❌ {f}  MISSING")

✅ /content/zip_ss_draws.csv  (377.0 MB)
✅ /content/zinb_ss_draws.csv  (377.0 MB)
✅ /content/zip_ss_summary.csv  (2.1 MB)
✅ /content/zinb_ss_summary.csv  (2.1 MB)
✅ /content/dengue_draws_v3.csv  (942.7 MB)


In [ ]:
import pandas as pd

for name in ['zip_ss', 'zinb_ss', 'hnb_ss']:
    print(f"\n{'='*60}")
    print(f"{name} diagnostics")
    print('='*60)
    s = pd.read_csv(f'/content/{name}_summary.csv', index_col=0)
    for p in ['beta0_mu', 'rho_zeta', 'sigma_zeta', 'sigma_eta']:
        if p in s.index:
            row = s.loc[p]
            print(f"  {p:12s}: mean={row['Mean']:8.3f}  "
                  f"Rhat={row['R_hat']:.4f}  ESS={row['ESS_bulk']:.0f}")
    if 'phi' in s.index:
        row = s.loc['phi']
        print(f"  {'phi':12s}: mean={row['Mean']:8.3f}  "
              f"Rhat={row['R_hat']:.4f}  ESS={row['ESS_bulk']:.0f}")


zip_ss diagnostics
  beta0_mu    : mean=  -0.222  Rhat=1.0078  ESS=428
  rho_zeta    : mean=   0.974  Rhat=1.0055  ESS=290
  sigma_zeta  : mean=   0.542  Rhat=1.0137  ESS=157
  sigma_eta   : mean=   1.439  Rhat=1.0731  ESS=29

zinb_ss diagnostics
  beta0_mu    : mean=  -0.245  Rhat=1.0108  ESS=230
  rho_zeta    : mean=   0.973  Rhat=1.0021  ESS=409
  sigma_zeta  : mean=   0.540  Rhat=1.0018  ESS=377
  sigma_eta   : mean=   1.103  Rhat=1.0143  ESS=26
  phi         : mean=  29.999  Rhat=1.0017  ESS=982

hnb_ss diagnostics
  beta0_mu    : mean=   0.791  Rhat=1.0028  ESS=836
  rho_zeta    : mean=   0.965  Rhat=1.0044  ESS=672
  sigma_zeta  : mean=   0.440  Rhat=1.0009  ESS=851
  sigma_eta   : mean=   1.466  Rhat=1.0024  ESS=784
  phi         : mean=  30.367  Rhat=1.0005  ESS=1439


In [ ]:
# CELL 7: WAIC COMPUTATION AND COMPARISON
import numpy as np
import pandas as pd

def waic_from_loglik(log_lik_draws):
    """
    log_lik_draws: array of shape (S, M)
        Each row is a posterior draw, each column is an observation.
    Returns: (waic, lppd, p_waic)
    """
    # lppd = sum_m log( mean_s exp(log_lik_sm) )
    max_ll = log_lik_draws.max(axis=0, keepdims=True)
    exp_shifted = np.exp(log_lik_draws - max_ll)
    lppd = float(np.sum(max_ll.squeeze() + np.log(exp_shifted.mean(axis=0))))

    # p_waic = sum_m var_s(log_lik_sm)
    p_waic = float(np.sum(log_lik_draws.var(axis=0, ddof=1)))

    waic = -2 * (lppd - p_waic)
    return waic, lppd, p_waic

def extract_loglik_from_csv(csv_path, M):
    """Extract log_lik columns from a draws CSV."""
    df = pd.read_csv(csv_path)
    cols = [c for c in df.columns if c.startswith('log_lik[')]
    # Sort by index
    idx = [int(c.replace('log_lik[', '').replace(']', '')) for c in cols]
    order = np.argsort(idx)
    return df[[cols[i] for i in order]].values

# --- HZMPS: load from the empirical notebook's draws CSV ---

try:
    ll_hzmps = extract_loglik_from_csv('/content/dengue_draws_v3.csv', M)
    waic_hzmps, lppd_h, p_h = waic_from_loglik(ll_hzmps)
    print(f"HZMPS: WAIC={waic_hzmps:.1f}  lppd={lppd_h:.1f}  p_waic={p_h:.1f}")
except FileNotFoundError:
    print("HZMPS draws CSV not found. Please upload dengue_draws_v3.csv to /content/.")
    waic_hzmps = None

# --- ZIP, ZINB, HNB: extract from their draws ---
ll_zip  = extract_loglik_from_csv('/content/zip_ss_draws.csv',  M)
ll_zinb = extract_loglik_from_csv('/content/zinb_ss_draws.csv', M)
ll_hnb  = extract_loglik_from_csv('/content/hnb_ss_draws.csv',  M)

waic_zip,  lppd_z,  p_z  = waic_from_loglik(ll_zip)
waic_zinb, lppd_zi, p_zi = waic_from_loglik(ll_zinb)
waic_hnb,  lppd_hb, p_hb = waic_from_loglik(ll_hnb)

print(f"ZIP : WAIC={waic_zip:.1f}   lppd={lppd_z:.1f}   p_waic={p_z:.1f}")
print(f"ZINB: WAIC={waic_zinb:.1f}  lppd={lppd_zi:.1f}  p_waic={p_zi:.1f}")
print(f"HNB : WAIC={waic_hnb:.1f}   lppd={lppd_hb:.1f}  p_waic={p_hb:.1f}")

# --- Comparison table ---
rows = []
if waic_hzmps is not None:
    rows.append({'Model': 'HZMPS-SS', 'WAIC': waic_hzmps,
                 'lppd': lppd_h, 'p_waic': p_h})
rows += [
    {'Model': 'ZIP-SS',  'WAIC': waic_zip,  'lppd': lppd_z,  'p_waic': p_z},
    {'Model': 'ZINB-SS', 'WAIC': waic_zinb, 'lppd': lppd_zi, 'p_waic': p_zi},
    {'Model': 'HNB-SS',  'WAIC': waic_hnb,  'lppd': lppd_hb, 'p_waic': p_hb},
]
table = pd.DataFrame(rows).sort_values('WAIC').reset_index(drop=True)
table['ΔWAIC'] = table['WAIC'] - table['WAIC'].min()
table.to_csv('/content/table_model_comparison.csv', index=False)

print("\n" + "="*60)
print("MODEL COMPARISON (lower WAIC is better)")
print("="*60)
print(table.to_string(index=False))

HZMPS: WAIC=43350.9  lppd=-19626.5  p_waic=2048.9
ZIP : WAIC=16466.0   lppd=-7001.1   p_waic=1231.9
ZINB: WAIC=17175.2  lppd=-7461.3  p_waic=1126.3
HNB : WAIC=19210.3   lppd=-8759.6  p_waic=845.5

MODEL COMPARISON (lower WAIC is better)
   Model         WAIC          lppd      p_waic        ΔWAIC
  ZIP-SS 16466.015262  -7001.072095 1231.935536     0.000000
 ZINB-SS 17175.197907  -7461.284253 1126.314700   709.182645
  HNB-SS 19210.255573  -8759.648137  845.479650  2744.240311
HZMPS-SS 43350.905606 -19626.521111 2048.931692 26884.890344
